In [ ]:
#@title 🚀 Bấm nút Play (▶) bên trái để bật VieNeu GPU Server
GPU_TIER = "Auto (Tự nhận T4 / L4 / A100)" #@param ["Auto (Tự nhận T4 / L4 / A100)", "T4 (Free/Pro - Batch 16)", "L4 (Colab Pro - Batch 32)", "A100 (Colab Pro+ - Batch 32)"]
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER = "STV-VieNeu" #@param {type:"string"}
EXPORT_FORMAT = "WAV (Không nén · Nhanh nhất)" #@param ["WAV (Không nén · Nhanh nhất)", "MP3 (128kbps · Nhẹ 9x)", "M4A (128kbps · Nhẹ 9x)"]
MODEL_SOURCE = "HF CDN Siêu tốc (Khuyên dùng · ~5-10s)" #@param ["HF CDN Siêu tốc (Khuyên dùng · ~5-10s)", "Cache Google Drive (hf_cache.tar)"]

import os, re, sys, time, json, glob, shutil, compileall, threading, subprocess, urllib.request
import importlib, importlib.metadata, importlib.util
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from IPython.display import HTML, clear_output, display

t_total0 = time.perf_counter()
timings = {"drive": 0.0, "prep": 0.0, "server": 0.0}

tier_map = {
    "Auto (Tự nhận T4 / L4 / A100)": "auto",
    "T4 (Free/Pro - Batch 16)": "T4",
    "L4 (Colab Pro - Batch 32)": "L4",
    "A100 (Colab Pro+ - Batch 32)": "A100",
}
fmt_map = {
    "WAV (Không nén · Nhanh nhất)": "wav",
    "MP3 (128kbps · Nhẹ 9x)": "mp3",
    "M4A (128kbps · Nhẹ 9x)": "m4a",
}
tier = tier_map.get(GPU_TIER, "auto")
export_fmt = fmt_map.get(EXPORT_FORMAT, "wav")
use_drive_model_cache = (MODEL_SOURCE == "Cache Google Drive (hf_cache.tar)")
safe_folder = re.sub(r'[<>:"/\\|?*\x00-\x1f]+', "_", str(DRIVE_FOLDER or "STV-VieNeu").strip().strip("/\\")) or "STV-VieNeu"

repo_dir = "/content/vieneu-tts"
runner_url = "https://raw.githubusercontent.com/trankhiembaolong/quick-novel-translator/main/colab/vieneu_colab_runner.py"
active_tmp_files = set()
active_procs = []
tunnel_proc = None
server_proc = None

def _preflight_tunnel():
    if not shutil.which("cloudflared"):
        subprocess.run(
            "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
            shell=True,
            check=True,
        )
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    active_procs.append(proc)
    url = ""
    deadline = time.time() + 60
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            time.sleep(0.1)
            continue
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
    def _drain_cf():
        try:
            for _ in proc.stdout:
                pass
        except Exception:
            pass
    threading.Thread(target=_drain_cf, daemon=True).start()
    return proc, url

def _preflight_repo():
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pnnbao97/VieNeu-TTS.git", repo_dir], check=True)
    rp = os.path.join(repo_dir, "vieneu_colab_runner.py")
    try:
        with urllib.request.urlopen(runner_url, timeout=15) as resp, open(rp, "wb") as out_f:
            out_f.write(resp.read())
        return rp
    except Exception as dl_err:
        print(f"⚠️ Không tải được runner mới ({dl_err}), dùng chế độ fallback mặc định.")
        return ""

print("⚡ [1/3] Đang kết nối Drive & chuẩn bị song song (cloudflared, repo, thư viện)...")
preflight_pool = ThreadPoolExecutor(max_workers=2)
tunnel_future = preflight_pool.submit(_preflight_tunnel)
repo_future = preflight_pool.submit(_preflight_repo)

drive_ready = False
cache_dir = ""
export_dir = ""
drive_display_dir = ""

if USE_GOOGLE_DRIVE:
    t_drv = time.perf_counter()
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            print("☁️ Đang kết nối Google Drive (bấm Cho phép / Allow nếu có hộp thoại)...")
            drive.mount("/content/drive", force_remount=False)
        if os.path.isdir("/content/drive/MyDrive"):
            base_drive = os.path.join("/content/drive/MyDrive", safe_folder)
            cache_dir = os.path.join(base_drive, "Cache")
            export_dir = os.path.join(base_drive, "Exports")
            os.makedirs(cache_dir, exist_ok=True)
            os.makedirs(export_dir, exist_ok=True)
            for stale_tmp in glob.glob(os.path.join(cache_dir, "*.tmp")):
                try:
                    os.remove(stale_tmp)
                except OSError:
                    pass
            drive_display_dir = f"MyDrive/{safe_folder}/Exports"
            drive_ready = True
            print(f"✅ Đã kết nối Google Drive: {drive_display_dir}")
    except Exception as drv_err:
        print(f"⚠️ Không kết nối được Google Drive ({drv_err}), chạy chế độ tạm thời không lưu Drive.")
    timings["drive"] = time.perf_counter() - t_drv

def atomic_tar_pack(parent_dir, target_name, dest_tar):
    tmp_tar = dest_tar + ".tmp"
    active_tmp_files.add(tmp_tar)
    try:
        subprocess.run(["tar", "-cf", tmp_tar, "-C", parent_dir, target_name], check=True)
        os.replace(tmp_tar, dest_tar)
        return True
    except Exception as exc:
        print(f"⚠️ Bỏ qua lưu cache {os.path.basename(dest_tar)}: {exc}")
        return False
    finally:
        active_tmp_files.discard(tmp_tar)
        if os.path.exists(tmp_tar):
            try:
                os.remove(tmp_tar)
            except OSError:
                pass

def _norm_pkg(name):
    return re.sub(r"[-_.+]+", "-", str(name or "")).lower()

def _snapshot_pkgs():
    importlib.invalidate_caches()
    return {
        _norm_pkg(d.metadata["Name"]): d.version
        for d in importlib.metadata.distributions()
        if d.metadata and d.metadata["Name"]
    }

os_tag = "linux"
try:
    with open("/etc/os-release", encoding="utf-8") as _osf:
        for _osl in _osf:
            if _osl.startswith("VERSION_CODENAME="):
                os_tag = _osl.split("=", 1)[1].strip().strip('"') or "linux"
                break
except Exception:
    pass

try:
    _tv = importlib.metadata.version("torch").split("+")[0].split(".")
    _torch_tag = f"torch{_tv[0]}_{_tv[1]}" if len(_tv) >= 2 else "notorch"
except Exception:
    _torch_tag = "notorch"
env_tag = f"py{sys.version_info.major}{sys.version_info.minor}_{_torch_tag}"

espeak_dir = "/content/espeak_debs"
espeak_tar = os.path.join(cache_dir, f"espeak_debs_{os_tag}.tar") if drive_ready else ""
site_dir = "/content/vieneu_site"
site_ready_flag = os.path.join(site_dir, ".ready")
site_tar = os.path.join(cache_dir, f"site_{env_tag}.tar") if drive_ready else ""
hf_dir = "/content/hf_cache"
hf_tar = os.path.join(cache_dir, "hf_cache.tar") if drive_ready else ""

runner_path = repo_future.result()
preflight_pool.shutdown(wait=False)

need_save_espeak = False
need_save_site = False
site_restored = False
hf_was_cached = False
before_pkgs = {}

def _install_packages_online():
    global before_pkgs, need_save_site
    before_pkgs = _snapshot_pkgs()
    pkgs = ["-e", repo_dir, "fastapi", "uvicorn", "huggingface_hub", "tokenizers", "hf_transfer"]
    if shutil.which("uv"):
        uv_res = subprocess.run(["uv", "pip", "install", "--python", sys.executable, "-q", *pkgs])
        if uv_res.returncode != 0:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    need_save_site = bool(drive_ready and site_tar)

def _prep_espeak():
    global need_save_espeak
    if not shutil.which("espeak-ng"):
        if espeak_tar and os.path.isfile(espeak_tar):
            try:
                shutil.rmtree(espeak_dir, ignore_errors=True)
                subprocess.run(["tar", "-xf", espeak_tar, "-C", "/content"], check=True)
                debs = sorted(glob.glob(os.path.join(espeak_dir, "*.deb")))
                if debs:
                    subprocess.run(["dpkg", "-i", *debs], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
            except Exception:
                pass
        if not shutil.which("espeak-ng"):
            shutil.rmtree(espeak_dir, ignore_errors=True)
            os.makedirs(espeak_dir, exist_ok=True)
            subprocess.run(
                f"apt-get update -qq && apt-get install -y -qq --download-only -o Dir::Cache::archives={espeak_dir} espeak-ng > /dev/null 2>&1",
                shell=True,
                check=False,
            )
            shutil.rmtree(os.path.join(espeak_dir, "partial"), ignore_errors=True)
            lock_f = os.path.join(espeak_dir, "lock")
            if os.path.exists(lock_f):
                try:
                    os.remove(lock_f)
                except OSError:
                    pass
            debs = sorted(glob.glob(os.path.join(espeak_dir, "*.deb")))
            if debs:
                subprocess.run(["dpkg", "-i", *debs], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
            if not shutil.which("espeak-ng"):
                subprocess.run("apt-get install -y -qq -f > /dev/null 2>&1 || apt-get install -y -qq espeak-ng > /dev/null 2>&1", shell=True, check=True)
            if drive_ready and espeak_tar and debs and shutil.which("espeak-ng"):
                need_save_espeak = True
    if export_fmt in ("mp3", "m4a") and not shutil.which("ffmpeg"):
        subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1", shell=True, check=True)

def _prep_site():
    global site_restored
    if os.path.isfile(site_ready_flag):
        site_restored = True
        return
    if site_tar and os.path.isfile(site_tar):
        try:
            print("⚡ Đang giải nén cache thư viện Python từ Google Drive...")
            shutil.rmtree(site_dir, ignore_errors=True)
            subprocess.run(["tar", "-xf", site_tar, "-C", "/content"], check=True)
            if os.path.isdir(site_dir) and any(os.scandir(site_dir)):
                Path(site_ready_flag).touch()
                site_restored = True
                return
        except Exception as site_err:
            print(f"⚠️ Không đọc được cache thư viện ({site_err}), chuyển sang cài trực tuyến...")
            shutil.rmtree(site_dir, ignore_errors=True)
    print("⏳ Đang cài đặt thư viện VieNeu-TTS (lần đầu)...")
    _install_packages_online()

def _prep_hf():
    global hf_was_cached
    if use_drive_model_cache and hf_tar and os.path.isfile(hf_tar):
        try:
            print("⚡ Đang nạp cache Model AI từ Google Drive...")
            shutil.rmtree(hf_dir, ignore_errors=True)
            subprocess.run(["tar", "-xf", hf_tar, "-C", "/content"], check=True)
            if os.path.isdir(hf_dir) and any(os.scandir(hf_dir)):
                hf_was_cached = True
        except Exception as hf_err:
            print(f"⚠️ Không đọc được cache Model AI ({hf_err}), sẽ tải mới...")
            shutil.rmtree(hf_dir, ignore_errors=True)
    os.makedirs(hf_dir, exist_ok=True)

t_prep0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as prep_pool:
    f_esp = prep_pool.submit(_prep_espeak)
    f_sit = prep_pool.submit(_prep_site)
    f_hfc = prep_pool.submit(_prep_hf)
    f_esp.result()
    f_sit.result()
    f_hfc.result()
timings["prep"] = time.perf_counter() - t_prep0

def _build_server_env(use_site_overlay, offline_hf):
    e = os.environ.copy()
    e["HOST"] = "127.0.0.1"
    e["PORT"] = "8000"
    e["VIENEU_COLAB_GPU_TIER"] = tier
    e["VIENEU_WATERMARK"] = "0"
    e["PYTHONUNBUFFERED"] = "1"
    e["HF_HOME"] = hf_dir
    e["HF_HUB_DISABLE_TELEMETRY"] = "1"
    if use_site_overlay:
        old_pp = e.get("PYTHONPATH", "")
        overlay_pp = f"{site_dir}:{repo_dir}/src:{repo_dir}"
        e["PYTHONPATH"] = f"{overlay_pp}:{old_pp}" if old_pp else overlay_pp
    has_hf_transfer = bool(
        importlib.util.find_spec("hf_transfer")
        or (use_site_overlay and glob.glob(os.path.join(site_dir, "hf_transfer*")))
    )
    if offline_hf:
        e["HF_HUB_OFFLINE"] = "1"
    elif has_hf_transfer:
        e["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    if drive_ready:
        e["VIENEU_DRIVE_EXPORT_DIR"] = export_dir
        e["VIENEU_DRIVE_EXPORT_DISPLAY"] = drive_display_dir
        e["VIENEU_DRIVE_EXPORT_FORMAT"] = export_fmt
    return e

def _launch_server(use_site_overlay, offline_hf):
    log_f = open("/content/vieneu_server.log", "w", encoding="utf-8")
    cmd = [sys.executable, runner_path] if (runner_path and os.path.exists(runner_path)) else [sys.executable, "-m", "apps.openai_speech"]
    proc = subprocess.Popen(cmd, cwd=repo_dir, env=_build_server_env(use_site_overlay, offline_hf), stdout=log_f, stderr=subprocess.STDOUT)
    return proc, log_f

print("⏳ [2/3] Đang khởi động VieNeu GPU Server & nạp model AI...")
t_srv0 = time.perf_counter()
server_proc, server_log = _launch_server(site_restored, hf_was_cached)
retried_online_pkg = False
health_data = {}
deadline = time.time() + 600
while time.time() < deadline:
    if server_proc.poll() is not None:
        server_log.flush()
        err_tail = open("/content/vieneu_server.log", encoding="utf-8", errors="ignore").read()[-2500:]
        if site_restored and not retried_online_pkg and any(k in err_tail for k in ("ModuleNotFoundError", "ImportError", "OSError")):
            print("⚠️ Cache thư viện chưa tương thích phiên bản Colab mới, tự động cài lại...")
            retried_online_pkg = True
            site_restored = False
            shutil.rmtree(site_dir, ignore_errors=True)
            if site_tar and os.path.isfile(site_tar):
                try:
                    os.remove(site_tar)
                except OSError:
                    pass
            _install_packages_online()
            server_log.close()
            server_proc, server_log = _launch_server(False, hf_was_cached)
            continue
        raise RuntimeError("VieNeu server thoát sớm. Log:\n" + err_tail)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as resp:
            if resp.status == 200:
                health_data = json.loads(resp.read().decode("utf-8"))
                break
    except Exception:
        pass
    time.sleep(0.4)
timings["server"] = time.perf_counter() - t_srv0

if not health_data:
    raise TimeoutError("VieNeu server khởi động quá 10 phút.")

print("⚡ [3/3] Đang hoàn tất link Cloudflare HTTPS Tunnel...")
tunnel_proc, tunnel_url = tunnel_future.result(timeout=60)
if not tunnel_url:
    raise RuntimeError("Không lấy được link trycloudflare.com từ cloudflared.")

if drive_ready and (need_save_espeak or need_save_site or (use_drive_model_cache and hf_tar and not hf_was_cached)):
    def _bg_save_caches():
        saved_any = False
        if need_save_espeak and os.path.isdir(espeak_dir) and glob.glob(os.path.join(espeak_dir, "*.deb")):
            shutil.rmtree(os.path.join(espeak_dir, "partial"), ignore_errors=True)
            lf = os.path.join(espeak_dir, "lock")
            if os.path.exists(lf):
                try:
                    os.remove(lf)
                except OSError:
                    pass
            if atomic_tar_pack("/content", "espeak_debs", espeak_tar):
                saved_any = True
        if need_save_site and site_tar:
            try:
                importlib.invalidate_caches()
                after_dists = {}
                for d in importlib.metadata.distributions():
                    if d.metadata and d.metadata["Name"]:
                        after_dists[_norm_pkg(d.metadata["Name"])] = d
                delta_names = {
                    n for n, d in after_dists.items()
                    if n not in ("vieneu", "vieneu-tts") and before_pkgs.get(n) != d.version
                }
                if delta_names:
                    shutil.rmtree(site_dir, ignore_errors=True)
                    os.makedirs(site_dir, exist_ok=True)
                    for pkg_name in delta_names:
                        dist = after_dists[pkg_name]
                        for rel in (dist.files or []):
                            rel_str = rel.as_posix()
                            if rel_str.startswith("..") or os.path.isabs(rel_str) or "__pycache__" in rel.parts or rel_str.endswith(".pyc"):
                                continue
                            src_p = Path(dist.locate_file(rel))
                            if src_p.is_file():
                                dst_p = Path(site_dir) / rel
                                dst_p.parent.mkdir(parents=True, exist_ok=True)
                                shutil.copy2(src_p, dst_p)
                    compileall.compile_dir(site_dir, quiet=1)
                    Path(site_ready_flag).touch()
                    if atomic_tar_pack("/content", "vieneu_site", site_tar):
                        saved_any = True
            except Exception as site_save_err:
                print(f"⚠️ Bỏ qua đóng gói cache thư viện: {site_save_err}", flush=True)
        if use_drive_model_cache and hf_tar and not hf_was_cached and os.path.isdir(hf_dir) and any(os.scandir(hf_dir)):
            if atomic_tar_pack("/content", "hf_cache", hf_tar):
                saved_any = True
        if saved_any:
            print("💾 Đã cập nhật cache Google Drive (từ lần sau khởi động ~25-45s)!", flush=True)
    threading.Thread(target=_bg_save_caches, daemon=True).start()

total_sec = time.perf_counter() - t_total0
gpu_label = health_data.get("gpu_name", "Colab GPU")
batch_label = health_data.get("batch_max_size", health_data.get("export_max_streams", 16))
dtype_label = health_data.get("model_dtype", "auto")
runtime_warn_html = ""
if tier in ("L4", "A100") and "T4" in gpu_label.upper():
    runtime_warn_html = f"<p style=\"margin: 8px 0 0; padding: 6px 10px; border-radius: 6px; background: #fef3c7; color: #92400e; font-size: 12px;\">💡 Bạn đang chọn <b>{tier}</b> nhưng Colab đang cấp phần cứng <b>{gpu_label}</b>. Để đổi lên L4/A100, bấm <b>Runtime → Change runtime type</b>.</p>"
drive_badge_html = (
    f"<p style=\"margin: 8px 0 0; padding: 6px 10px; border-radius: 6px; background: #d1fae5; color: #065f46; font-size: 12px;\">☁️ <b>Drive Cache & Lưu trực tiếp:</b> <code>{drive_display_dir}</code> ({export_fmt.upper()})</p>"
    if drive_ready
    else ""
)
timing_badge_html = f"<p style=\"margin: 8px 0 0; padding: 6px 10px; border-radius: 6px; background: #ecfdf5; border: 1px solid #a7f3d0; color: #047857; font-size: 12px;\">⏱️ Khởi động <b>{total_sec:.1f}s</b> · Drive {timings['drive']:.1f}s · Cài nền {timings['prep']:.1f}s · Model+GPU {timings['server']:.1f}s</p>"

clear_output(wait=True)
card_html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 620px; border: 2px solid #10b981; border-radius: 14px; padding: 18px 20px; background: linear-gradient(135deg, #ecfdf5 0%, #f0fdf4 100%); color: #064e3b;">
  <div style="display: flex; align-items: center; justify-content: space-between; gap: 10px; flex-wrap: wrap;">
    <div style="font-size: 16px; font-weight: 800;">🟢 VieNeu Colab GPU đã sẵn sàng!</div>
    <span style="background: #059669; color: #fff; padding: 4px 10px; border-radius: 999px; font-size: 12px; font-weight: 700;">{gpu_label} · Batch {batch_label} · {dtype_label}</span>
  </div>
  <p style="margin: 10px 0 12px; font-size: 13.5px; color: #065f46;">Bấm nút bên dưới để copy link, sau đó quay lại Extension Quick Novel Translator và bấm <b>📋 Dán</b> rồi <b>⬇ Tải Audio (WAV)</b>.</p>
  <div style="display: flex; gap: 8px; align-items: center; flex-wrap: wrap;">
    <input id="stv-colab-url" type="text" readonly value="{tunnel_url}" style="flex: 1; min-width: 240px; padding: 10px 12px; border: 1px solid #34d399; border-radius: 8px; font-size: 13px; font-weight: 600; background: #fff; color: #0f172a;" />
    <button id="stv-copy-btn" onclick="(function(){{var inp=document.getElementById('stv-colab-url');var btn=document.getElementById('stv-copy-btn');var done=function(){{btn.innerText='✓ Đã Copy! Quay lại Extension bấm Dán';btn.style.background='#047857';}};if(navigator.clipboard&&navigator.clipboard.writeText){{navigator.clipboard.writeText(inp.value).then(done).catch(function(){{inp.select();document.execCommand('copy');done();}});}}else{{inp.select();document.execCommand('copy');done();}}}})()" style="padding: 11px 18px; border: none; border-radius: 8px; background: #10b981; color: #fff; font-size: 14px; font-weight: 700; cursor: pointer;">📋 Copy Link Kết Nối</button>
  </div>
  {drive_badge_html}
  {timing_badge_html}
  {runtime_warn_html}
  <p style="margin: 10px 0 0; font-size: 12px; color: #047857;">⚠️ Giữ tab Colab này mở trong lúc Extension đang tạo Audio.</p>
</div>
"""
display(HTML(card_html))

log_pos = os.path.getsize("/content/vieneu_server.log") if os.path.exists("/content/vieneu_server.log") else 0
try:
    while server_proc.poll() is None and tunnel_proc.poll() is None:
        if os.path.exists("/content/vieneu_server.log"):
            with open("/content/vieneu_server.log", "r", encoding="utf-8", errors="ignore") as lf:
                lf.seek(log_pos)
                chunk = lf.read()
                log_pos = lf.tell()
            for log_line in chunk.splitlines():
                if "BATCH done:" in log_line or "BATCH sent:" in log_line or "💾 DRIVE saved:" in log_line or "ERROR" in log_line:
                    print(log_line, flush=True)
        time.sleep(2)
except KeyboardInterrupt:
    pass
finally:
    for p in set(active_procs + [tunnel_proc, server_proc]):
        if p is not None and p.poll() is None:
            p.terminate()
    for tmp_f in list(active_tmp_files):
        if os.path.exists(tmp_f):
            try:
                os.remove(tmp_f)
            except OSError:
                pass
